In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.facecolor"] = "white"
%matplotlib inline

In [ ]:
df = pd.read_csv("AmesHousing.csv")
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
# فحص الـ Outliers بصريًا قبل أي تنظيف (Boxplot)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ["SalePrice", "Lot Area", "Gr Liv Area"]):
    sns.boxplot(y=df[col], ax=ax, color="steelblue")
    ax.set_title(col)
fig.suptitle("Outlier Check (before any cleaning)", fontsize=14)
plt.tight_layout()
plt.savefig("fig_outliers_raw.png", dpi=150)
plt.show()

for col in ["SalePrice", "Lot Area", "Gr Liv Area"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col:15s} -> {n_outliers} outlier(s) خارج حدود IQR ({n_outliers/len(df)*100:.1f}% من البيانات)")

In [ ]:
num_cols = list(set(df.describe().columns.to_list()) - {"PID", "Order"})
missing_counts = df[num_cols].isnull().sum()
print("Numeric columns with missing values:")
print(missing_counts[missing_counts > 0])

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(9, 8))
sns.barplot(x=missing_pct.values, y=missing_pct.index, hue=missing_pct.index, palette="Reds_r", legend=False)
plt.title("Missing Values per Column (%)")
plt.xlabel("% missing")
plt.ylabel("")
plt.axvline(50, color="black", linestyle="--", linewidth=1, label="50% threshold")
plt.legend()
plt.tight_layout()
plt.savefig("fig_missing_values.png", dpi=150)
plt.show()

In [ ]:
# Mas Vnr Area: impute with median (skewed continuous variable)
df["Mas Vnr Area"] = df["Mas Vnr Area"].fillna(df["Mas Vnr Area"].median())

In [ ]:
# Garage Yr Blt: fix a known data-entry typo (year 2207 -> 2007),
# then impute missing years only for houses that actually HAVE a garage.
# Houses with no garage keep Garage Yr Blt as NaN (handled via Has_Garage).
df.loc[df["Garage Yr Blt"] == 2207, "Garage Yr Blt"] = 2007
has_garage = df["Garage Type"].notna()
median_garage_yr = df.loc[has_garage, "Garage Yr Blt"].median()
df.loc[has_garage & df["Garage Yr Blt"].isnull(), "Garage Yr Blt"] = median_garage_yr
df["Has_Garage"] = has_garage.astype(int)

In [ ]:
# Lot Frontage: impute with median
df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Lot Frontage"].median())

In [ ]:
# Remaining numeric columns: a missing value genuinely means "0" of that
# feature (e.g. no basement -> 0 sqft of basement), not an unknown value.
cols_to_fill_zero = [
    "Bsmt Half Bath", "Bsmt Full Bath", "BsmtFin SF 1", "BsmtFin SF 2",
    "Total Bsmt SF", "Bsmt Unf SF", "Garage Cars", "Garage Area",
]
df[cols_to_fill_zero] = df[cols_to_fill_zero].fillna(0)

In [ ]:
cat_cols = df.select_dtypes(exclude=["number"]).columns
missing_cat = df[cat_cols].isnull().sum()
print("Categorical columns with missing values:")
print(missing_cat[missing_cat > 0])

In [ ]:
# Electrical missing -> fill with the mode (most common system).
# All other categorical NaNs mean "feature doesn't exist" -> "NA".
mode_col = ["Electrical"]
none_cols = [
    "Alley", "Mas Vnr Type", "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
    "BsmtFin Type 1", "BsmtFin Type 2", "Fireplace Qu", "Garage Type",
    "Garage Finish", "Garage Qual", "Garage Cond", "Pool QC", "Fence", "Misc Feature",
]
for col in mode_col:
    df[col] = df[col].fillna(df[col].mode()[0])
for col in none_cols:
    df[col] = df[col].fillna("NA")

print("Total remaining missing values:", df.isnull().sum().sum())

In [ ]:
cols_to_binary = ["Pool QC", "Misc Feature", "Alley", "Fence"]
for col in cols_to_binary:
    df[col] = df[col].apply(lambda x: 0 if x == "NA" else 1)
    df.rename(columns={col: f"Has_{col}"}, inplace=True)

df[["Has_Pool QC", "Has_Misc Feature", "Has_Alley", "Has_Fence"]].head()

In [ ]:
# رسمة بتوضح ليه القرار ده كان صح: أغلب البيوت فعليًا "مالهاش" الخاصية دي
cols_to_binary_check = ["Has_Pool QC", "Has_Misc Feature", "Has_Alley", "Has_Fence"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, cols_to_binary_check):
    vc = df[col].map({0: "Doesn't have", 1: "Has it"}).value_counts()
    sns.barplot(x=vc.index, y=vc.values, ax=ax, hue=vc.index, palette="pastel", legend=False)
    ax.set_title(col)
    ax.set_ylabel("Number of houses")
    for i, v in enumerate(vc.values):
        ax.text(i, v, str(v), ha="center", va="bottom")
fig.suptitle("Why convert to binary flags? Most houses simply don't have this feature", fontsize=13)
plt.tight_layout()
plt.savefig("fig_has_flags.png", dpi=150)
plt.show()

In [ ]:
num_cols = list(set(df.describe().columns.to_list()) - {"PID", "Order"})
cat_cols = list(set(df.columns.to_list()) - set(num_cols) - {"PID", "Order"})


def review_low_variance_columns(
    df, num_cols, cat_cols, threshold=0.8, price_gap_pct=10,
    protected_cols=None, min_group_size=20,
):
    """
    Flags and drops columns that are dominated by a single value AND show
    no meaningful relationship with SalePrice.

    - Numeric columns: dropped if |correlation with SalePrice| < 0.1.
    - Categorical columns: dropped if the price gap between the dominant
      category and the rest is below `price_gap_pct`, OR if the minority
      group is too small (< min_group_size) to trust the comparison.

    Returns the filtered df, the list of dropped columns, and a decision
    log (used later for the summary chart).
    """
    protected_cols = protected_cols or []
    cols_to_drop = []
    decision_log = []

    for col in df.columns:
        if col in ["Order", "PID", "SalePrice"] + protected_cols:
            continue

        top_freq = df[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq <= threshold:
            continue

        if col in num_cols:
            corr = abs(df[col].corr(df["SalePrice"]))
            decision = "DROP" if corr < 0.1 else "KEEP"
            reason = f"corr={corr:.3f}"
        else:
            dominant_value = df[col].value_counts(dropna=False).idxmax()
            dominant_mask = df[col] == dominant_value
            other_mask = ~dominant_mask

            if other_mask.sum() < min_group_size:
                decision = "DROP"
                reason = f"minority group too small (n={other_mask.sum()})"
            else:
                dominant_price = df.loc[dominant_mask, "SalePrice"].mean()
                other_price = df.loc[other_mask, "SalePrice"].mean()
                price_gap = abs(dominant_price - other_price) / df["SalePrice"].mean() * 100
                decision = "DROP" if price_gap < price_gap_pct else "KEEP"
                reason = f"price gap={price_gap:.1f}%"

        decision_log.append({"column": col, "decision": decision, "dominance": top_freq, "reason": reason})
        print(f"{decision}  {col:20s} | dominance={top_freq:.1%} | {reason}")
        if decision == "DROP":
            cols_to_drop.append(col)

    return df.drop(columns=cols_to_drop), cols_to_drop, pd.DataFrame(decision_log)

In [ ]:
protected = ["Has_Pool QC", "Has_Misc Feature", "Has_Alley", "Has_Fence", "Has_Garage"]
df, dropped_cols, decision_log = review_low_variance_columns(df, num_cols=num_cols, cat_cols=cat_cols, protected_cols=protected)
print(f"\nTotal low-variance columns dropped: {len(dropped_cols)}")
print("Shape after low-variance filtering:", df.shape)

In [ ]:
# ملخص بصري لقرارات الفنكشن (كام عمود اتشال وكام اتساب، ومتوسط نسبة السيطرة لكل حالة)
summary = decision_log["decision"].value_counts()
plt.figure(figsize=(6, 4))
sns.barplot(x=summary.index, y=summary.values, hue=summary.index,
            palette={"KEEP": "seagreen", "DROP": "indianred"}, legend=False)
plt.title("Low-Variance Review: Columns Kept vs Dropped")
plt.ylabel("Number of columns")
for i, v in enumerate(summary.values):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig("fig_lowvariance_summary.png", dpi=150)
plt.show()

In [ ]:
num_cols = list(set(df.describe().columns.to_list()) - {"PID", "Order"})
corr_matrix = df[num_cols].corr()
corr_matrix_no_target = corr_matrix.drop(["SalePrice"], axis=0).drop(["SalePrice"], axis=1)

REDUNDANCY_THRESHOLD = 0.8  # standard cutoff for "strongly correlated"
cols_to_drop_redundant = set()
cols = corr_matrix_no_target.columns.tolist()

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        col_i, col_j = cols[i], cols[j]
        corr_value = corr_matrix_no_target.loc[col_i, col_j]
        if abs(corr_value) > REDUNDANCY_THRESHOLD:
            corr_i_target = abs(corr_matrix.loc[col_i, "SalePrice"])
            corr_j_target = abs(corr_matrix.loc[col_j, "SalePrice"])
            weaker_col = col_j if corr_i_target >= corr_j_target else col_i
            cols_to_drop_redundant.add(weaker_col)
            print(f"{col_i:20s} <-> {col_j:20s} | corr={corr_value:.3f} -> dropping '{weaker_col}'")

df = df.drop(columns=list(cols_to_drop_redundant))
print(f"\nRedundant columns dropped: {sorted(cols_to_drop_redundant)}")
print("Shape after removing redundant features:", df.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=df, x="Total Bsmt SF", y="Gr Liv Area", ax=axes[0], alpha=0.4)
axes[0].set_title("Sanity check: still-kept features are not duplicates")
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, ax=axes[1], cbar_kws={"shrink": 0.6})
axes[1].set_title("Full correlation matrix (reference)")
plt.tight_layout()
plt.savefig("fig_redundancy_sanity.png", dpi=150)
plt.show()

In [ ]:
id_cols_dropped = [c for c in ["Order", "PID"] if c in df.columns]
df = df.drop(columns=id_cols_dropped)
print(f"Dropped identifier columns: {id_cols_dropped}")
print("Shape after dropping identifiers:", df.shape)

In [ ]:
df_clean_pre_encoding = df.copy()
assert "Neighborhood" in df_clean_pre_encoding.columns, "Pipeline order issue: re-run from the top."
print("Snapshot OK —", df_clean_pre_encoding.shape)

In [ ]:
# كام قيمة فريدة في كل عمود nominal قبل الـ One-Hot؟ (ده بيفسر ليه عدد الأعمدة هيزيد كتير)
remaining_cat_cols_preview = df.select_dtypes(exclude=["number"]).columns.tolist()
ordinal_like = ["Exter Qual","Exter Cond","Bsmt Qual","Bsmt Cond","Heating QC","Kitchen Qual",
                "Fireplace Qu","Garage Qual","Garage Cond","Bsmt Exposure","BsmtFin Type 1",
                "BsmtFin Type 2","Garage Finish","Lot Shape","Land Slope","Functional",
                "Paved Drive","Street","Central Air"]
nominal_preview = [c for c in remaining_cat_cols_preview if c not in ordinal_like]

cardinality = df[nominal_preview].nunique().sort_values(ascending=False)
plt.figure(figsize=(9, 6))
sns.barplot(x=cardinality.values, y=cardinality.index, hue=cardinality.index, palette="crest", legend=False)
plt.title("Number of unique categories per nominal column (before One-Hot)")
plt.xlabel("Unique categories")
plt.tight_layout()
plt.savefig("fig_nominal_cardinality.png", dpi=150)
plt.show()

In [ ]:
# 8a. Ordinal quality scales (Po < Fa < TA < Gd < Ex) -> ordered integers.
quality_map = {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
quality_cols = [c for c in [
    "Exter Qual", "Exter Cond", "Bsmt Qual", "Bsmt Cond",
    "Heating QC", "Kitchen Qual", "Fireplace Qu", "Garage Qual", "Garage Cond",
] if c in df.columns]
for col in quality_cols:
    df[col] = df[col].map(quality_map)

print("Quality columns encoded:", quality_cols)

In [ ]:
# 8b. Other columns with a domain-specific order.
ordinal_maps = {
    "Bsmt Exposure": {"NA": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFin Type 1": {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFin Type 2": {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "Garage Finish": {"NA": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Lot Shape": {"IR3": 0, "IR2": 1, "IR1": 2, "Reg": 3},
    "Land Slope": {"Sev": 0, "Mod": 1, "Gtl": 2},
    "Functional": {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
    "Paved Drive": {"N": 0, "P": 1, "Y": 2},
}
for col, mapping in ordinal_maps.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

print("Ordinal columns encoded:", list(ordinal_maps.keys()))

In [ ]:
# 8c. True binary columns -> 0/1 directly (no need for One-Hot).
binary_maps = {"Street": {"Grvl": 0, "Pave": 1}, "Central Air": {"N": 0, "Y": 1}}
for col, mapping in binary_maps.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

In [ ]:
# 8d. Remaining categorical columns = nominal (no inherent order) -> One-Hot.
# drop_first=True avoids the dummy-variable trap (redundant baseline column).
remaining_cat_cols = df.select_dtypes(exclude=["number"]).columns.tolist()
print(f"Nominal columns one-hot encoded ({len(remaining_cat_cols)}):", remaining_cat_cols)

df = pd.get_dummies(df, columns=remaining_cat_cols, drop_first=True)
print("Shape after encoding:", df.shape)

In [ ]:
cols_to_exclude_from_scaling = [
    "SalePrice",                                                              # target: real units for reporting
    "Overall Qual", "Overall Cond",                                           # small-range ordinal ratings
    "Mo Sold", "Yr Sold",                                                     # calendar fields
    "Has_Alley", "Has_Fence", "Has_Pool QC", "Has_Misc Feature", "Has_Garage",  # binary flags
]
current_numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
cols_to_scale = [c for c in current_numeric_cols if c not in cols_to_exclude_from_scaling]

scaler = RobustScaler()
df_scaled = df.copy()
df_scaled[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print(f"Columns scaled: {len(cols_to_scale)}")
print("Final clean (real-units) df shape :", df.shape)
print("Final scaled df shape             :", df_scaled.shape)

In [ ]:
median_val = df["Lot Area"].median()
typical_idx = (df["Lot Area"] - median_val).abs().idxmin()
lot_area_raw = df[["Lot Area"]].reset_index(drop=True)

scalers_test = {"Min-Max": MinMaxScaler(), "Z-Score (Standard)": StandardScaler(), "Robust": RobustScaler()}

before_vals = {name: s.fit_transform(lot_area_raw).flatten()[typical_idx] for name, s in scalers_test.items()}

lot_area_with_outlier = pd.concat([lot_area_raw, pd.DataFrame({"Lot Area": [2_000_000]})], ignore_index=True)
after_vals = {name: s.fit_transform(lot_area_with_outlier).flatten()[typical_idx] for name, s in scalers_test.items()}

shifts = {name: abs(after_vals[name] - before_vals[name]) for name in scalers_test}

plt.figure(figsize=(8, 5))
bars = plt.bar(shifts.keys(), shifts.values(), color=["indianred", "goldenrod", "seagreen"])
plt.title("Stability Test: How much does a TYPICAL house's\nscaled value shift after ONE new extreme outlier appears?", fontsize=13)
plt.ylabel("Shift in scaled value")
for bar, v in zip(bars, shifts.values()):
    plt.text(bar.get_x() + bar.get_width()/2, v, f"{v:.4f}", ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig("fig_scaler_stability.png", dpi=150)
plt.show()

for name, shift in shifts.items():
    print(f"{name:20s} -> shift = {shift:.4f}")

In [ ]:
print("ملخص الـ Outliers حسب قاعدة IQR (للمعرفة فقط — لم يتم الحذف):\n")
for col in ["SalePrice", "Lot Area", "Gr Liv Area"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col:15s} | {n_outliers:4d} outlier(s) ({n_outliers/len(df)*100:4.1f}%) | bounds=({lower:,.0f}, {upper:,.0f}) | max={df[col].max():,.0f}")

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["SalePrice"], kde=True, bins=40, color="steelblue")
plt.title("Distribution of Sale Price", fontdict={"size": 16})
plt.xlabel("Sale Price ($)")
plt.ylabel("Number of Houses")
plt.axvline(df["SalePrice"].mean(), color="red", linestyle="--",
            label=f"Mean = ${df['SalePrice'].mean():,.0f}")
plt.axvline(df["SalePrice"].median(), color="green", linestyle="--",
            label=f"Median = ${df['SalePrice'].median():,.0f}")
plt.legend()
plt.tight_layout()
plt.savefig("fig_01_saleprice_distribution.png", dpi=150)
plt.show()

In [ ]:
corr_with_price = (
    df.corr(numeric_only=True)["SalePrice"]
    .drop("SalePrice")
    .sort_values(key=abs, ascending=False)
)
top_15 = corr_with_price.head(15)

plt.figure(figsize=(9, 7))
bar_colors = ["seagreen" if v > 0 else "indianred" for v in top_15.values]
sns.barplot(x=top_15.values, y=top_15.index, hue=top_15.index,
            palette=bar_colors, legend=False)
plt.title("Top 15 Features Correlated with Sale Price", fontdict={"size": 15})
plt.xlabel("Correlation Coefficient")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig("fig_02_top_correlations.png", dpi=150)
plt.show()

In [ ]:
strongest_feature = corr_with_price.index[0]
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x=strongest_feature, y="SalePrice", alpha=0.4, color="teal")
sns.regplot(data=df, x=strongest_feature, y="SalePrice", scatter=False, color="darkred")
plt.title(f"SalePrice vs {strongest_feature}", fontdict={"size": 15})
plt.xlabel(strongest_feature)
plt.ylabel("Sale Price ($)")
plt.tight_layout()
plt.savefig("fig_03_strongest_relationship.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))
sns.boxplot(data=df, x="Overall Qual", y="SalePrice", hue="Overall Qual",
            palette="coolwarm", legend=False)
plt.title("Sale Price by Overall Quality Rating", fontdict={"size": 15})
plt.xlabel("Overall Quality (1 = Poor, 10 = Excellent)")
plt.ylabel("Sale Price ($)")
plt.tight_layout()
plt.savefig("fig_04_price_by_quality.png", dpi=150)
plt.show()

In [ ]:
neighborhood_avg = (
    df_clean_pre_encoding.groupby("Neighborhood")["SalePrice"]
    .mean()
    .sort_values(ascending=False)
)
plt.figure(figsize=(10, 9))
sns.barplot(x=neighborhood_avg.values, y=neighborhood_avg.index,
            hue=neighborhood_avg.index, palette="viridis", legend=False)
plt.title("Average Sale Price by Neighborhood", fontdict={"size": 15})
plt.xlabel("Average Sale Price ($)")
plt.ylabel("Neighborhood")
plt.tight_layout()
plt.savefig("fig_05_price_by_neighborhood.png", dpi=150)
plt.show()

In [ ]:
top_features = corr_with_price.head(11).index.tolist() + ["SalePrice"]
plt.figure(figsize=(10, 8))
sns.heatmap(df[top_features].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap - Top Features Only")
plt.tight_layout()
plt.savefig("fig_06_top_features_heatmap.png", dpi=150)
plt.show()

In [ ]:
df_clean_pre_encoding.to_csv("ames_clean_readable.csv", index=False)
df.to_csv("ames_clean_encoded.csv", index=False)
df_scaled.to_csv("ames_clean_encoded_scaled.csv", index=False)

print("تم حفظ 3 ملفات:")
print(" - ames_clean_readable.csv       (قابلة للقراءة، قبل الـ Encoding)", df_clean_pre_encoding.shape)
print(" - ames_clean_encoded.csv        (نهائية، قيم حقيقية)           ", df.shape)
print(" - ames_clean_encoded_scaled.csv (نهائية، مطبّعة)                ", df_scaled.shape)

In [ ]:
top_neighborhoods = neighborhood_avg.head(3)
bottom_neighborhoods = neighborhood_avg.tail(3)
skew_direction = "right-skewed (a few high-priced homes pull the average up)" \
    if df["SalePrice"].mean() > df["SalePrice"].median() else "left-skewed"

print("=" * 70)
print("KEY FINDINGS")
print("=" * 70)
print(f"""
1. Price distribution:
   SalePrice is {skew_direction}.
   Mean = ${df['SalePrice'].mean():,.0f} | Median = ${df['SalePrice'].median():,.0f}

2. Strongest price drivers (by correlation with SalePrice):
{top_15.head(5).to_string()}

3. Overall Quality has a strong, monotonic relationship with price
   (correlation = {corr_with_price['Overall Qual']:.2f}) -> each additional
   quality point is associated with a clear, consistent price premium.

4. Highest-priced neighborhoods:
{top_neighborhoods.to_string()}

5. Lowest-priced neighborhoods:
{bottom_neighborhoods.to_string()}
""")

In [ ]:
print("=" * 70)
print("BUSINESS RECOMMENDATIONS")
print("=" * 70)
print(f"""
- Renovation ROI: Since Overall Quality, Kitchen Qual and Exter Qual are
  all in the top correlated features, quality-focused renovations
  (kitchen, exterior finish) are likely to yield the strongest price lift
  relative to size-only additions.
- Investment targeting: {top_neighborhoods.index[0]}, {top_neighborhoods.index[1]}
  and {top_neighborhoods.index[2]} command the highest average prices and
  may offer the best resale positioning for premium listings.
- Value opportunities: {bottom_neighborhoods.index[0]} and
  {bottom_neighborhoods.index[1]} show the lowest average prices and could
  be worth investigating for undervalued-entry investment opportunities.
- Next step: given the skew in SalePrice, a log-transform of the target
  should be tested before any regression modeling, to stabilize variance.
""")